## Some resources:
1. LLVM-tutor: https://github.com/banach-space/llvm-tutor/blob/main/lib/OpcodeCounter.cpp
2. LLVM source code: https://github.com/llvm/llvm-project/tree/main/llvm/include/llvm/IR
3. UFMG videos - https://www.youtube.com/@compilerslab/videos

## Typical workflow:

- Set up environment variables so that clang, opt, and llvm-config are found.
- Write the pass source code into a .cpp file.
- Compile the pass into a .dylib. (.dylib = your LLVM pass as a plugin, opt loads it at runtime and applies your transformation.)
- Create a test C file, lower it to LLVM IR (.ll).
- Run opt with your pass and capture the output.

In [1]:
import os

# Adjust to your LLVM Homebrew install path
LLVM_HOME = "/opt/homebrew/Cellar/llvm/20.1.8"

os.environ["PATH"] = f"{LLVM_HOME}/bin:" + os.environ["PATH"]
os.environ["DYLD_LIBRARY_PATH"] = f"{LLVM_HOME}/lib:" + os.environ.get("DYLD_LIBRARY_PATH", "")

!which clang
!which opt
!llvm-config --version

/opt/homebrew/Cellar/llvm/20.1.8/bin/clang
/opt/homebrew/Cellar/llvm/20.1.8/bin/opt
20.1.8


In [2]:
pass_code = r"""
#include "llvm/IR/Function.h"
#include "llvm/IR/PassManager.h"
#include "llvm/Passes/PassBuilder.h"
#include "llvm/Passes/PassPlugin.h"
#include "llvm/Support/raw_ostream.h"
#include "llvm/IR/Instructions.h"
#include "llvm/IR/IRBuilder.h"

using namespace llvm;

struct MulByTwoToAddPass : public PassInfoMixin<MulByTwoToAddPass> {
    // Main entry point, takes IR unit to run the pass on (&F) and the
    // corresponding pass manager (to be queried if need be)
    PreservedAnalyses run(Function &F, FunctionAnalysisManager &) {
        bool Changed = false;

        for (auto &BB : F) {
            for (auto Inst = BB.begin(), E = BB.end(); Inst != E;) {
                Instruction *I = &*Inst++;
                
                // Look for binary operators
                if (auto *BinOp = dyn_cast<BinaryOperator>(I)) {
                    if (BinOp->getOpcode() == Instruction::Mul) {
                        Value *Op1 = BinOp->getOperand(0);
                        Value *Op2 = BinOp->getOperand(1);

                        // Case: x * 2
                        if (auto *C = dyn_cast<ConstantInt>(Op2)) {
                            if (C->equalsInt(2)) {
                                IRBuilder<> Builder(BinOp);
                                Value *NewAdd = Builder.CreateAdd(Op1, Op1, "mul_by_2_as_add");
                                BinOp->replaceAllUsesWith(NewAdd);
                                errs() << "[MulByTwoToAddPass] Replaced '"
                                       << *BinOp << "' with '" << *NewAdd
                                       << "' in function " << F.getName() << "\n";
                                BinOp->eraseFromParent();
                                Changed = true;
                                continue;
                            }
                        }

                        // Case: 2 * x
                        if (auto *C = dyn_cast<ConstantInt>(Op1)) {
                            if (C->equalsInt(2)) {
                                IRBuilder<> Builder(BinOp);
                                Value *NewAdd = Builder.CreateAdd(Op2, Op2, "mul_by_2_as_add");
                                BinOp->replaceAllUsesWith(NewAdd);
                                errs() << "[MulByTwoToAddPass] Replaced '"
                                       << *BinOp << "' with '" << *NewAdd
                                       << "' in function " << F.getName() << "\n";
                                BinOp->eraseFromParent();
                                Changed = true;
                                continue;
                            }
                        }
                    }
                }
            }
        }

        if (Changed)
            return PreservedAnalyses::none();
        return PreservedAnalyses::all();
    }

    // Without isRequired returning true, this pass will be skipped for functions
    // decorated with the optnone LLVM attribute. Note that clang -O0 decorates
    // all functions with optnone.
    static bool isRequired() { return true; }
}; //Note!! close struct properly with semicolon!


// Register plugin 
// This is the core interface for pass plugins. It guarantees that 'opt' will
// be able to recognize MulByTwoToAddPass when added to the pass pipeline on the
// command line, i.e. via '-passes=MulByTwoToAddPass'
extern "C" LLVM_ATTRIBUTE_WEAK PassPluginLibraryInfo llvmGetPassPluginInfo() {
    return {
    //-----------------------------------------------------------------------------
    // New PM Registration
    //-----------------------------------------------------------------------------
        LLVM_PLUGIN_API_VERSION, "MulByTwoToAddPass", "v0.1",
        [](PassBuilder &PB) {
            PB.registerPipelineParsingCallback(
                [](StringRef Name, FunctionPassManager &FPM,
                   ArrayRef<PassBuilder::PipelineElement>) {
                    if (Name == "mul-to-add") {
                        FPM.addPass(MulByTwoToAddPass());
                        return true;
                    }
                    return false;
                });
        }
    };
}

"""

with open("llvm_passes/MulByTwoToAddPass.cpp", "w") as f:
    f.write(pass_code)


In [3]:
c_code = r"""
int mul2(int x) {
    return x * 2;
}

int main() {
    int y = mul2(10);
    return y;
}
"""

with open("llvm_passes/mulby2.c", "w") as f:
    f.write(c_code)

In [4]:
!clang -emit-llvm -S -O0 llvm_passes/mulby2.c -o llvm_passes/mulby2.ll

In [5]:
#-shared → build a shared library
# -fPIC → generate position-independent code (required for shared libs)
# MulByTwoToAddPasss.dylib → the output dynamic library
# When you build your LLVM pass as a plugin, you’re actually compiling it into a shared library 
# that can be dynamically loaded by opt (LLVM’s optimizer tool).
!clang++ -std=c++17 -fPIC -shared llvm_passes/MulByTwoToAddPass.cpp -o llvm_passes/MulByTwoToAddPass.dylib \
    `llvm-config --cxxflags --ldflags --system-libs --libs core passes`


In [6]:
!opt -load-pass-plugin=./llvm_passes/MulByTwoToAddPass.dylib \
    -passes="function(mul-to-add)" \
    -S llvm_passes/mulby2.ll -o llvm_passes/mulby2_opt.ll

[MulByTwoToAddPass] Replaced '  %4 = mul nsw i32 %3, 2' with '  %mul_by_2_as_add = add i32 %3, %3' in function mul2


In [7]:
cat llvm_passes/mulby2_opt.ll

; ModuleID = 'llvm_passes/mulby2.ll'
source_filename = "llvm_passes/mulby2.c"
target datalayout = "e-m:o-p270:32:32-p271:32:32-p272:64:64-i64:64-i128:128-n32:64-S128-Fn32"
target triple = "arm64-apple-macosx15.0.0"

; Function Attrs: noinline nounwind optnone ssp uwtable(sync)
define i32 @mul2(i32 noundef %0) #0 {
  %2 = alloca i32, align 4
  store i32 %0, ptr %2, align 4
  %3 = load i32, ptr %2, align 4
  %mul_by_2_as_add = add i32 %3, %3
  ret i32 %mul_by_2_as_add
}

; Function Attrs: noinline nounwind optnone ssp uwtable(sync)
define i32 @main() #0 {
  %1 = alloca i32, align 4
  %2 = alloca i32, align 4
  store i32 0, ptr %1, align 4
  %3 = call i32 @mul2(i32 noundef 10)
  store i32 %3, ptr %2, align 4
  %4 = load i32, ptr %2, align 4
  ret i32 %4
}

attributes #0 = { noinline nounwind optnone ssp uwtable(sync) "frame-pointer"="non-leaf" "no-trapping-math"="true" "stack-protector-buffer-size"="8" "target-cpu"="apple-m1" "target-features"="+aes,+altnzcv,+ccdp,+ccidx,+ccpp,+complxnum,